# Exercise 5.1 — OTOCs and the Failure of Clifford Chaos

**Chapter 5: Unitary Designs** &nbsp;|&nbsp; *Section 5.2: Clifford Circuits and Discrete Scrambling*

---

## Motivation

Clifford circuits form exact unitary 3-designs — they reproduce the first three moments of the Haar measure.  This makes them extremely useful for benchmarking and error correction.  Yet they are **efficiently classically simulable** (Gottesman-Knill theorem) and therefore cannot exhibit true quantum chaos.

The OTOC $C(t)$ quantifies scrambling: in chaotic systems, it grows exponentially as $\sim e^{2\lambda t}$ before saturating.  This exercise shows that Clifford evolution can only produce $C(t) \in \{0, 4\}$ — a discrete jump, never a smooth exponential.  This is the fundamental obstruction to defining a Lyapunov exponent for Clifford dynamics.

## Preliminaries / Toolbox

Before diving into the solution, recall the following concepts:

**1. Pauli Group:** The group generated by $I, X, Y, Z$ with phase factors. Elements either commute $[A, B]=0$ or anticommute $\{A, B\}=0$.

**2. Clifford Group:** The normalizer of the Pauli group. A unitary $U$ is Clifford if $U P U^\dagger$ is another Pauli operator $P'$, for any Pauli $P$.

**3. Traces of Paulis:** All non-identity Pauli operators are traceless, $\mathrm{Tr}(P)=0$, and square to identity $P^2=I$.

**4. OTOC bounds:** Because continuous scrambling $\sim e^{\lambda t}$ implies smooth interpolation, a strictly binary-valued squared commutator means true (Lyapunov) chaos cannot occur.



## Exercise Statement

Let $U$ be a Clifford unitary on $N$ qubits.  For single-qubit Paulis $W$ and $V$, define the squared commutator:

$$
C(t) = \frac{1}{D}\mathrm{Tr}\bigl([W(t), V]^\dagger [W(t), V]\bigr), \qquad W(t) = U^\dagger W U.
$$

Prove $C(t) \in \{0, 4\}$ and conclude that Clifford circuits have no Lyapunov exponent.

## Solution

### Step 1: Clifford conjugation maps Paulis to Paulis

By definition, a Clifford unitary $U$ normalizes the Pauli group: $U^\dagger P U \in \mathcal{P}_N$ (up to a sign $\pm 1, \pm i$) for any Pauli $P$.  In particular, $W(t) = U^\dagger W U$ is itself a Pauli operator (possibly on many qubits).

### Step 2: Paulis either commute or anticommute

Any two elements of the Pauli group on $N$ qubits either **commute** or **anticommute** — there is no intermediate possibility.  This is because Pauli operators are tensor products of $\{I, X, Y, Z\}$, and each pair of single-qubit Paulis either commutes (if they are equal or one is $I$) or anticommutes (if they are different non-identity Paulis).  The full commutation is the product of the per-site factors.

### Step 3: The two cases

**Case 1:** $[W(t), V] = 0$ (they commute).

$$
C(t) = 0.
$$

**Case 2:** $\{W(t), V\} = 0$ (they anticommute).  Then $[W(t), V] = 2W(t)V$.

$$
C(t) = \frac{1}{D}\mathrm{Tr}\bigl(4V^\dagger W(t)^\dagger W(t) V\bigr) = \frac{4}{D}\mathrm{Tr}(I) = 4.
$$

(We used $W(t)^\dagger W(t) = I$ and $V^\dagger V = I$ since Paulis are unitary.)

### Step 4: No Lyapunov exponent

$$
\boxed{C(t) \in \{0, 4\} \quad \text{for all Clifford } U.}
$$

Continuous exponential growth $C(t) \sim \epsilon\, e^{2\lambda t}$ requires the OTOC to pass through intermediate values — impossible with the discrete set $\{0, 4\}$.  Clifford circuits scramble (the Pauli $W$ can spread to a high-weight operator), but the scrambling is **instantaneous** once the light cone reaches $V$, not gradual.  This is why Cliffords are a 3-design (matching moments up to $k=3$) but **not** a good model of quantum chaos.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

# For Pauli operators P, Q:
# Either [P,Q] = 0 (commute) or {P,Q} = 0 (anticommute)
# C(P,Q) = <[P,Q]^dag[P,Q]> / D
# If [P,Q]=0: C = 0  => F = 1
# If {P,Q}=0: [P,Q] = 2PQ, C = 4*Tr(I)/D = 4  => F = -1
print('Clifford OTOC values:')
print('  Commuting Paulis:     C = 0, F = 1')
print('  Anticommuting Paulis: C = 4, F = -1')
print()
print('No intermediate values possible!')
print('This binary behavior is an obstruction to Lyapunov chaos.')
print('True chaos requires continuous F(t) ~ 1 - epsilon*e^{lambda*t}')
print('but Clifford dynamics can only produce F in {-1, +1}.')

---
## Numerical Verification

In [ ]:
import numpy as np
from itertools import product

I2 = np.eye(2, dtype=complex)
sx = np.array([[0,1],[1,0]], dtype=complex)
sy = np.array([[0,-1j],[1j,0]])
sz = np.array([[1,0],[0,-1]], dtype=complex)
H_gate = np.array([[1,1],[1,-1]], dtype=complex)/np.sqrt(2)
S_gate = np.array([[1,0],[0,1j]], dtype=complex)
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)

def random_clifford_2q():
    U = np.eye(4, dtype=complex)
    for _ in range(10):
        g = np.random.choice(3)
        if g == 0: U = np.kron(H_gate, I2) @ U
        elif g == 1: U = np.kron(S_gate, I2) @ U
        else: U = CNOT @ U
    return U

paulis_1q = [I2, sx, sy, sz]
N, D = 2, 4

print("C(t) values for 200 random 2-qubit Cliffords:")
C_values = set()
for _ in range(200):
    U = random_clifford_2q()
    for W_1q in [sx, sy, sz]:
        for V_1q in [sx, sy, sz]:
            W = np.kron(W_1q, I2)
            V = np.kron(I2, V_1q)
            Wt = U.conj().T @ W @ U
            comm = Wt @ V - V @ Wt
            C = np.trace(comm.conj().T @ comm).real / D
            C_values.add(round(C, 6))

print(f"  Distinct C values: {sorted(C_values)}")
assert C_values == {0.0, 4.0} or C_values <= {0.0, 4.0}
print("  C(t) ∈ {0, 4} confirmed: no Lyapunov exponent. ✓")